# GrainSTEM · Strain mapping

Intragrain strain mapping from a 4D-STEM NBD dataset (py4DSTEM). Reflection
positions are localized from the **autocorrelation** of each nanobeam pattern and
refined with **subpixel upsampling**; the strain tensor (εxx, εyy, εxy) and lattice
rotation (θ) are then fitted against a grain-interior reference lattice.

**Usage:** set `filepath_data` in the configuration cell below to your dataset, then
run top to bottom. Adjust the ROI crop, the selected NMF component, and the
basis-vector indices (g1, g2) as annotated in the corresponding cells.

See `README.md` in this folder for the full method description.

In [ ]:
# ==============================================================================
# Complete 4D-STEM Strain Analysis Notebook (Headless/SSH Compatible)
# ==============================================================================
import os
import h5py
import numpy as np
import py4DSTEM
import tifffile
import matplotlib.pyplot as plt
from os.path import splitext
from matplotlib.colors import Normalize
from IPython.display import Image, display

# GPU Setting
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# ------------------------------------------------------------------------------
# 1. Define File Paths & Preprocessing (Binning)
# ------------------------------------------------------------------------------
dataset_name = "Strain_acq2"
filepath_data = f"./data/{dataset_name}.raw"   # <-- EDIT: path to your .raw/.h5 dataset
filepath_save = splitext(filepath_data)[0] + '_preprocessed_filtered.h5'
save_prefix = f"./results/{dataset_name}_"

import os
os.makedirs("./results", exist_ok=True)

R_BIN = 2   # real-space binning factor (scan)
Q_BIN = 1   # reciprocal-space binning factor (diffraction)

# Load datacube (supports HDF5, DM4, and RAW formats)
if filepath_data.endswith(".raw"):
    datacube = py4DSTEM.import_file(filepath_data, EMPAD_shape=(256, 256, 130, 128))
elif filepath_data.endswith(".dm4"):
    datacube = py4DSTEM.import_file(filepath_data)
else:
    datacube = py4DSTEM.read(filepath_data)

# Bin and preprocess
datacube.bin_R(R_BIN)
datacube.bin_Q(Q_BIN)
print(f"Datacube loaded. Raw scanner dimensions: {datacube.data.shape[:2]}")

In [ ]:
# ------------------------------------------------------------------------------
# 2. Real Space Crop (ROI) Tuning & Preview
# ------------------------------------------------------------------------------
# Set the real-space crop boundaries (Rx_min, Rx_max, Ry_min, Ry_max)
# Rx corresponds to the scan columns (X), Ry corresponds to the scan rows (Y)
# You can run this cell repeatedly to adjust the crop boundaries!
rx_min, rx_max = 50,110
ry_min, ry_max = 0,60

# 1) Calculate the original virtual BF image for visual context
Q_Ny, Q_Nx = datacube.data.shape[2], datacube.data.shape[3]
cy, cx = Q_Ny // 2, Q_Nx // 2
y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
bf_mask = np.sqrt((y_grid - cy)**2 + (x_grid - cx)**2) <= 15.0

orig_vbf = np.sum(datacube.data[:, :, bf_mask], axis=2)

# 2) Perform crop on a COPY of the datacube to keep the original intact for re-running
cropped_cube = datacube.copy()
cropped_cube.crop_R((ry_min, ry_max, rx_min, rx_max))
print(f"Original scan shape: {datacube.data.shape[:2]}")
print(f"Cropped scan shape: {cropped_cube.data.shape[:2]}")

# 3) Calculate the cropped virtual BF image
cropped_vbf = np.sum(cropped_cube.data[:, :, bf_mask], axis=2)

# 4) Visualize the crop region and the cropped result
fig_crop, axes_crop = plt.subplots(1, 2, figsize=(12, 6))

# Original v-BF with bounding box
axes_crop[0].imshow(orig_vbf, cmap='gray')
rect = plt.Rectangle((rx_min, ry_min), rx_max - rx_min, ry_max - ry_min, 
                     fill=False, color='red', linewidth=2, linestyle='--')
axes_crop[0].add_patch(rect)
axes_crop[0].set_title("Original Virtual BF (with Crop Box)")
axes_crop[0].axis('on')

# Cropped v-BF
axes_crop[1].imshow(cropped_vbf, cmap='gray')
axes_crop[1].set_title("Cropped Virtual BF (ROI)")
axes_crop[1].axis('on')

plt.tight_layout()

# 5) Save a lightweight preview image and force display inline using IPython
save_path_crop = f"{save_prefix}crop_preview.png"
fig_crop.savefig(save_path_crop, bbox_inches='tight', dpi=150)
plt.close(fig_crop)

display(Image(filename=save_path_crop))


In [ ]:
# ------------------------------------------------------------------------------
# 3. Save Cropped Datacube
# ------------------------------------------------------------------------------
# Run this cell only when you are satisfied with the crop boundaries in the cell above.
import numpy as np, gc

# 1) Finalize the cropped data as a contiguous (C-contiguous) array independent of the original
#    -> safe to delete the original afterward, and no extra copy is made when saving with h5py
cropped_cube.data = np.ascontiguousarray(cropped_cube.data)

# 2) Remove the original datacube (frees the ~4GB held by the full cube)
if 'datacube' in globals() and datacube is not cropped_cube:
    del datacube
gc.collect()

# 3) Now rename cropped_cube to datacube and save
datacube = cropped_cube
py4DSTEM.save(filepath_save, datacube, mode='o')
print(f"Cropped datacube saved to: {filepath_save}")

In [ ]:
# ------------------------------------------------------------------------------
# 4. Mask Out Central Beam & Threshold Data (Pre-NMF)
# ------------------------------------------------------------------------------
# Create a copy of the cropped data for modification
modified_data = np.array(datacube.data[:, :], dtype=np.float32)

# Apply circular mask to zero out the central direct beam
mask_center_y = 63.5  # Match Part1_ROI-Pick exactly
mask_center_x = 62.5  # Match Part1_ROI-Pick exactly
mask_radius = 14.0  # Match Part1_ROI-Pick exactly

y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
dist_from_center = np.sqrt((y_grid - mask_center_y)**2 + (x_grid - mask_center_x)**2)
circular_mask = dist_from_center <= mask_radius

modified_data[:, :, circular_mask] = 0

# Apply binary thresholding to isolate Bragg peak intensities
binary_threshold = 1000  # Match Part1_ROI-Pick exactly (5500 instead of 100)
modified_data = np.where(modified_data > binary_threshold, 1, 0)
modified_data = np.nan_to_num(modified_data, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Circular mask (Center: {mask_center_y}, {mask_center_x}, Radius: {mask_radius}) applied successfully.")
print(f"Binary threshold ({binary_threshold}) applied. Shape of processed data: {modified_data.shape}")


In [ ]:
# ------------------------------------------------------------------------------
# 5. Run CPU NMF (numpy) & Plot Features and Components
# ------------------------------------------------------------------------------
import matplotlib.cm as cm

# Set the number of NMF components directly (Change this value if needed!)
n_components = 5

def fit_transform_nmf_cpu(X, n_components=20, max_iter=1000):
    print("Using device: cpu (numpy)")

    X_mat = np.array(X, dtype=np.float32)  # explicit copy to preserve input
    M, N = X_mat.shape
    K = n_components

    rng = np.random.default_rng()
    W = (rng.random((M, K)) + 0.1).astype(np.float32)
    H = (rng.random((K, N)) + 0.1).astype(np.float32)
    eps = 1e-9

    for i in range(max_iter):
        Wt = W.T
        H *= (Wt @ X_mat) / ((Wt @ W) @ H + eps)
        Ht = H.T
        W *= (X_mat @ Ht) / (W @ (H @ Ht) + eps)

        if (i + 1) % 100 == 0 or i == 0:
            reconstruction_err = np.linalg.norm(X_mat - W @ H)
            print(f"Iteration {i+1}/{max_iter} - Reconstruction Error: {reconstruction_err:.4f}")

    return W, H

flattened_data = modified_data.reshape((-1, modified_data.shape[2] * modified_data.shape[3]))
flattened_data = np.nan_to_num(flattened_data, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

W, H = fit_transform_nmf_cpu(flattened_data, n_components=n_components, max_iter=1000)
print("NMF fit completed successfully.")

norm = Normalize(vmin=W.min(), vmax=W.max())
cmap = plt.colormaps['hsv'].resampled(n_components)
colors = [cmap(i)[:3] for i in range(n_components)]

combined_image = np.zeros((modified_data.shape[0], modified_data.shape[1], 3))
fig, axes = plt.subplots(2, n_components, figsize=(3 * n_components, 6))

for i in range(n_components):
    raw_component = H[i].reshape(modified_data.shape[2], modified_data.shape[3])
    exponent = 0.3
    normalized_component = (raw_component - raw_component.min()) / (raw_component.max() - raw_component.min())
    component_image = np.power(normalized_component, exponent)

    feature_image = np.zeros((modified_data.shape[0], modified_data.shape[1], 3))
    for j in range(3):
        feature_image[:, :, j] = W[:, i].reshape(modified_data.shape[0], modified_data.shape[1]) * colors[i][j]

    if n_components == 1:
        ax0 = axes[0]
        ax1 = axes[1]
    else:
        ax0 = axes[0, i]
        ax1 = axes[1, i]

    ax0.imshow(feature_image)
    ax0.set_title(f'Feature Image {i+1}')
    ax0.axis('off')

    rgb_component = np.zeros((modified_data.shape[2], modified_data.shape[3], 3))
    for j in range(3):
        rgb_component[:, :, j] = component_image * colors[i][j]
    ax1.imshow(rgb_component)
    ax1.set_title(f'Component Image {i+1}')
    ax1.axis('off')

    combined_image += feature_image

plt.tight_layout()

# Save and force display using IPython display
save_path_grid = f"{save_prefix}nmf_grid.png"
fig.savefig(save_path_grid, bbox_inches='tight', dpi=150)
plt.close(fig)

display(Image(filename=save_path_grid))

combined_image = np.clip(combined_image, 0, 1)

In [ ]:
# ------------------------------------------------------------------------------
# 6. Select NMF Components & Generate ROI Mask
# ------------------------------------------------------------------------------
# Enter the NMF component numbers (1-based index) you want to extract as your ROI
# For example, selected_nmf_components = [1, 3]
selected_nmf_components = [4]  # Adjust this list based on the NMF grid plot above!

# Threshold for creating the mask from the normalized feature maps (0.0 ~ 1.0)
mask_threshold = 0.60

# 1) Generate the combined mask
selected_indices = [idx - 1 for idx in selected_nmf_components]
H_scan, W_scan = modified_data.shape[0], modified_data.shape[1]
combined_mask = np.zeros((H_scan, W_scan), dtype=bool)

for idx in selected_indices:
    if 0 <= idx < n_components:
        feature_img = W[:, idx].reshape(H_scan, W_scan)
        # Normalize feature map to [0, 1]
        norm_feat = (feature_img - feature_img.min()) / (feature_img.max() - feature_img.min() + 1e-8)
        # Combine with thresholding
        combined_mask |= (norm_feat >= mask_threshold)
    else:
        print(f"Warning: Component index {idx+1} is out of bounds (Total: {n_components})")

print(f"Generated combined mask for NMF components: {selected_nmf_components}")
print(f"Active mask area: {np.sum(combined_mask)} pixels out of {H_scan*W_scan} total scan positions.")

# 2) Visualize the generated mask
fig_mask, ax = plt.subplots(figsize=(6, 6))
ax.imshow(combined_mask, cmap='gray')
ax.set_title(f"Generated Mask (Components: {selected_nmf_components}, Thresh: {mask_threshold})")
ax.axis('off')

# Save and force display inline via IPython
save_path_mask = f"{save_prefix}selected_mask.png"
fig_mask.savefig(save_path_mask, bbox_inches='tight', dpi=150)
plt.close(fig_mask)

display(Image(filename=save_path_mask))


In [ ]:
# ------------------------------------------------------------------------------
# 7. Self Cross-Correlation & Resize ROI Diffraction Patterns to 256x256
# ------------------------------------------------------------------------------
from scipy.ndimage import zoom
try:
    import cv2
    has_cv2 = True
except ImportError:
    has_cv2 = False

print("Calculating self cross-correlation and resizing to 256x256 for ROI pixels...")

H_scan, W_scan = datacube.data.shape[0], datacube.data.shape[1]
Q_Ny, Q_Nx = datacube.data.shape[2], datacube.data.shape[3]

# Create a new array of shape (H_scan, W_scan, 256, 256)
new_dp_data = np.zeros((H_scan, W_scan, 256, 256), dtype=np.float32)

active_y, active_x = np.where(combined_mask)
total_active = len(active_y)

for idx in range(total_active):
    y, x = active_y[idx], active_x[idx]
    dp = datacube.data[y, x, :, :].astype(np.float32)
    
    # 1) Calculate self cross-correlation (autocorrelation) using FFT
    dp_ft = np.fft.fft2(dp)
    cc = np.fft.ifftshift(np.real(np.fft.ifft2(dp_ft * np.conjugate(dp_ft))))
    
    # 2) Resize to 256x256
    if has_cv2:
        cc_resized = cv2.resize(cc, (256, 256), interpolation=cv2.INTER_CUBIC)
    else:
        cc_resized = zoom(cc, (256 / Q_Ny, 256 / Q_Nx), order=3)
        
    new_dp_data[y, x, :, :] = cc_resized
    
    if (idx + 1) % 1000 == 0 or idx == 0:
        print(f"Processed {idx + 1}/{total_active} pixels...")

# Create a new datacube with the self-correlated and resized data
selected_datacube = py4DSTEM.DataCube(new_dp_data)

# Update dimensions metadata of the new datacube
selected_datacube.calibration.R_pixel_size = datacube.calibration.R_pixel_size
selected_datacube.calibration.Q_pixel_size = datacube.calibration.Q_pixel_size * (Q_Nx / 256.0)

print(f"New self-correlated and resized selected_datacube shape: {selected_datacube.data.shape}")


In [ ]:
# ------------------------------------------------------------------------------
# 8. Generate Probe Template from Middle Pixel of Real Space Scan
# ------------------------------------------------------------------------------
# Find coordinates of active pixels in the ROI mask to get a valid non-zero DP
active_y, active_x = np.where(combined_mask)
H_scan, W_scan = selected_datacube.data.shape[0], selected_datacube.data.shape[1]

if len(active_y) > 0:
    # Use a scan position near the middle of the active ROI region
    target_idx = len(active_y) // 2
    ref_y = active_y[target_idx]
    ref_x = active_x[target_idx]
    print(f"Extracting probe from active ROI scan position: ({ref_y}, {ref_x})")
else:
    ref_y = H_scan // 2
    ref_x = W_scan // 2
    print(f"Warning: No active ROI pixels found. Falling back to center: ({ref_y}, {ref_x})")

# Extract the DP at the selected pixel (autocorrelation, shape 256x256)
dp_mid = selected_datacube.data[ref_y, ref_x, :, :].astype(np.float32)
Q_Ny, Q_Nx = dp_mid.shape

# AUTOMATIC BEAM CENTER DETECTION:
# Search for the autocorrelation peak in a 40x40 window around the mathematical center (128, 128)
search_half = 20
cy_guess, cx_guess = Q_Ny // 2-2, Q_Nx // 2+1
window = dp_mid[cy_guess - search_half : cy_guess + search_half, 
                 cx_guess - search_half : cx_guess + search_half]
local_max_y, local_max_x = np.unravel_index(np.argmax(window), window.shape)

bf_center_y = cy_guess - search_half + local_max_y
bf_center_x = cx_guess - search_half + local_max_x
bf_radius = 12.0  # BF disk radius scaled up to 256x256 (originally 6.0)

# 1) Create a full-size template with the BF disk centered at (bf_center_y, bf_center_x)
y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
dist_from_center = np.sqrt((y_grid - bf_center_y)**2 + (x_grid - bf_center_x)**2)
bf_mask = dist_from_center <= bf_radius

template_centered = np.zeros_like(dp_mid)
template_centered[bf_mask] = dp_mid[bf_mask]

# Normalize the centered template
if np.sum(template_centered) > 0:
    template_centered = template_centered / np.sum(template_centered)

# 2) Shift the template origin to (0, 0) using np.roll
# This rolls the actual detected direct beam center exactly to index (0, 0)
# to eliminate any sub-pixel/pixel-level coordinate offset in the cross-correlation!
probe_kernel = np.roll(template_centered, shift=(-bf_center_y, -bf_center_x), axis=(0, 1))

print(f"Automatically detected direct beam center: ({bf_center_y}, {bf_center_x})")
print(f"Generated shifted probe template from middle pixel DP.")
print(f"Template shape: {probe_kernel.shape}, BF radius: {bf_radius} px.")

# Plot the generated probe kernel (We plot template_centered for clean visualization)
fig_probe, ax_probe = plt.subplots(1, 2, figsize=(10, 5))
ax_probe[0].imshow(dp_mid**0.3, cmap='inferno')
ax_probe[0].set_title("Original DP at Selected Pixel")
circle = plt.Circle((bf_center_x, bf_center_y), bf_radius, color='cyan', fill=False, linewidth=2, linestyle='--')
ax_probe[0].add_patch(circle)

ax_probe[1].imshow(template_centered**0.3, cmap='inferno')
ax_probe[1].set_title("Extracted Probe Template (Centered)")

# Save and force display inline
save_path_probe = f"{save_prefix}probe_template.png"
fig_probe.savefig(save_path_probe, bbox_inches='tight', dpi=150)
plt.close(fig_probe)

display(Image(filename=save_path_probe))


In [ ]:
# ------------------------------------------------------------------------------
# 9. Build Reference Reflection Passband (from central 3x3 average DP)
# ------------------------------------------------------------------------------
from scipy.ndimage import distance_transform_edt

if 'selected_datacube_clean_backup' not in globals():
    selected_datacube_clean_backup = selected_datacube.data.copy()
selected_datacube.data = selected_datacube_clean_backup.copy()

# --- Pass-radius parameter (fixed 6px) ---
KEEP_RADIUS = 5.0

# --- [NEW] Distance-band cutoff from the direct beam: peaks outside this band are excluded from the reference ---
# find_Bragg_disks itself has no radius-cutoff option (edgeBoundary is only relative to the image edge,
# not a distance cut relative to the direct beam), so we post-process filter by distance after detection.
# Adjust the value to match the diffraction pattern size (typically 256x256, center 128).
MAX_BEAM_DISTANCE = 120  # exclude peaks farther than this value (px) from the direct-beam center
MIN_BEAM_DISTANCE = 60    # [NEW] exclude peaks closer than this value (px) to the direct-beam center
                         #        (the direct beam itself is at distance 0, so it is always kept as an exception)

print("Finding the core pixel of the grain...")
dist_map = distance_transform_edt(combined_mask)
cy_core, cx_core = np.unravel_index(np.argmax(dist_map), dist_map.shape)
print(f"Grain core scan position detected at: Y={cy_core}, X={cx_core}")

H_scan, W_scan = selected_datacube.data.shape[0], selected_datacube.data.shape[1]
y_indices = np.clip([cy_core-1, cy_core, cy_core+1], 0, H_scan-1)
x_indices = np.clip([cx_core-1, cx_core, cx_core+1], 0, W_scan-1)

dp_list = []
for y in y_indices:
    for x in x_indices:
        if combined_mask[y, x]:
            dp_list.append(selected_datacube.data[y, x, :, :].astype(np.float32))

if len(dp_list) > 0:
    dp_core_avg = np.mean(dp_list, axis=0)
    print(f"Averaged diffraction patterns from {len(dp_list)} pixels in central 3x3 scan window.")
else:
    dp_core_avg = selected_datacube.data[cy_core, cx_core, :, :].astype(np.float32)
    print("Warning: Using single core pixel DP.")

detect_params_core = {
    'corrPower': 1.0,
    'sigma': 0,
    'edgeBoundary': 32,
    'minRelativeIntensity': 0.001,
    'minAbsoluteIntensity': 0.1,
    'minPeakSpacing': 35,
    'subpixel': 'poly',
    'upsample_factor': 8,
    'maxNumPeaks': 1000,
}

orig_dp = selected_datacube.data[cy_core, cx_core, :, :].copy()
selected_datacube.data[cy_core, cx_core, :, :] = dp_core_avg

print("Detecting reference peaks on the averaged DP...")
core_peaks = selected_datacube.find_Bragg_disks(
    data=([cy_core], [cx_core]),
    template=probe_kernel,
    **detect_params_core
)
selected_datacube.data[cy_core, cx_core, :, :] = orig_dp

ref_qx = np.asarray(core_peaks[0].data['qx'], dtype=np.float64)  # row direction
ref_qy = np.asarray(core_peaks[0].data['qy'], dtype=np.float64)  # col direction
print(f"Detected {len(ref_qx)} reference peaks (including direct beam).")

# 4) Identify the central direct beam
Q_Nx_dp, Q_Ny_dp = selected_datacube.data.shape[2], selected_datacube.data.shape[3]
center_qx, center_qy = Q_Nx_dp / 2.0, Q_Ny_dp / 2.0
dists_to_center = np.sqrt((ref_qx - center_qx)**2 + (ref_qy - center_qy)**2)
direct_beam_idx = int(np.argmin(dists_to_center))
beam_qx = ref_qx[direct_beam_idx]
beam_qy = ref_qy[direct_beam_idx]
print(f"Direct beam located at (qx=row, qy=col): ({beam_qx:.2f}, {beam_qy:.2f})")

# --- [NEW] 4b) Radius cutoff relative to the direct beam: exclude peaks outside the [MIN, MAX] band ---
dists_from_beam = np.sqrt((ref_qx - beam_qx)**2 + (ref_qy - beam_qy)**2)
within_radius = (dists_from_beam >= MIN_BEAM_DISTANCE) & (dists_from_beam <= MAX_BEAM_DISTANCE)

# The direct beam itself is at distance 0, so it would be caught by the MIN cut -> always keep it
within_radius[direct_beam_idx] = True

n_excluded = int((~within_radius).sum())
if n_excluded > 0:
    print(f"Excluding {n_excluded} peak(s) outside "
          f"[{MIN_BEAM_DISTANCE:.0f}, {MAX_BEAM_DISTANCE:.0f}]px band from the direct beam "
          f"(too close = beam halo/noise, too far = higher-order reflections).")
ref_qx = ref_qx[within_radius]
ref_qy = ref_qy[within_radius]

# Indices changed due to filtering, so recompute the direct-beam index
dists_to_center = dists_to_center[within_radius]
direct_beam_idx = int(np.argmin(dists_to_center))
beam_qx = ref_qx[direct_beam_idx]
beam_qy = ref_qy[direct_beam_idx]
print(f"Reference peaks after radius cutoff: {len(ref_qx)} "
      f"(within [{MIN_BEAM_DISTANCE:.0f}, {MAX_BEAM_DISTANCE:.0f}]px of direct beam).")

# 5) Build a 2D "passband KEEP mask"
qx_grid, qy_grid = np.ogrid[:Q_Nx_dp, :Q_Ny_dp]
keep_mask_Q = np.zeros((Q_Nx_dp, Q_Ny_dp), dtype=bool)

ref_circles = []
for i in range(len(ref_qx)):
    r = KEEP_RADIUS
    d = np.sqrt((qx_grid - ref_qx[i])**2 + (qy_grid - ref_qy[i])**2)
    keep_mask_Q[d <= r] = True
    ref_circles.append((float(ref_qx[i]), float(ref_qy[i]), float(r)))

n_keep = int(keep_mask_Q.sum())
print(f"Keep-mask built: {len(ref_circles)} circles (radius={KEEP_RADIUS}px), "
      f"{n_keep} px marked as valid ({100*n_keep/keep_mask_Q.size:.2f}% of DP area).")
print(">> The data was not masked. This keep_mask_Q is used for peak post-processing after Cell 14.")

# 6) [3-panel verification visualization]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(combined_mask, cmap='gray')
axes[0].plot(cx_core, cy_core, 'r*', markersize=15, label='Core Pixel')
axes[0].set_title(f"1. Real Space ROI & Core Pixel\n(Y={cy_core}, X={cx_core})")
axes[0].legend(loc='upper right')
axes[0].axis('on')

axes[1].imshow(dp_core_avg**0.1, cmap='inferno')
axes[1].set_title("2. Original 3x3 Average DP")
axes[1].axis('off')

axes[2].imshow(dp_core_avg**0.1, cmap='inferno')
axes[2].imshow(keep_mask_Q, cmap='gray', alpha=0.15)
for i in range(len(ref_qx)):
    color = 'cyan' if i == direct_beam_idx else 'red'
    circle = plt.Circle((ref_qy[i], ref_qx[i]), KEEP_RADIUS,
                        color=color, fill=False, linestyle='--', linewidth=2.0)
    axes[2].add_patch(circle)
# --- [NEW] Reference markers for the cutoff band: outer (yellow) / inner (light green) ---
cutoff_circle = plt.Circle((beam_qy, beam_qx), MAX_BEAM_DISTANCE,
                            color='yellow', fill=False, linestyle=':', linewidth=1.5)
axes[2].add_patch(cutoff_circle)
if MIN_BEAM_DISTANCE > 0:
    min_circle = plt.Circle((beam_qy, beam_qx), MIN_BEAM_DISTANCE,
                            color='lime', fill=False, linestyle=':', linewidth=1.5)
    axes[2].add_patch(min_circle)
axes[2].set_title(f"3. Overlay: Reference Peaks & {KEEP_RADIUS:.0f}px Keep-Mask\n"
                   f"(band=[{MIN_BEAM_DISTANCE:.0f}, {MAX_BEAM_DISTANCE:.0f}]px; "
                   f"yellow=max, lime=min)")
axes[2].axis('off')

plt.tight_layout()
save_path_diff_mask = f"{save_prefix}diffraction_passband_mask.png"
plt.savefig(save_path_diff_mask, bbox_inches='tight', dpi=150)
plt.close()
display(Image(filename=save_path_diff_mask))


# ------------------------------------------------------------------------------
# 9b. Helper: filter detected peaks so ONLY peaks inside the reference keep-mask survive
# ------------------------------------------------------------------------------
from emdfile import PointList

def _filter_pl_by_keepmask(pl, keep_mask):
    data = pl.data
    name = getattr(pl, 'name', '')
    if len(data) == 0:
        return PointList(np.array([], dtype=pl.dtype), name=name)
    xs = np.clip(np.round(data['qx']).astype(int), 0, keep_mask.shape[0] - 1)
    ys = np.clip(np.round(data['qy']).astype(int), 0, keep_mask.shape[1] - 1)
    inside = keep_mask[xs, ys]
    return PointList(data[inside].copy(), name=name)

def filter_disks_by_reference(disks, keep_mask=None):
    if keep_mask is None:
        keep_mask = keep_mask_Q
    if isinstance(disks, (list, tuple)):
        return [_filter_pl_by_keepmask(pl, keep_mask) for pl in disks]
    if isinstance(disks, PointList):
        return _filter_pl_by_keepmask(disks, keep_mask)
    return disks

print(f"Reference filter helper ready. keep_mask_Q active area: "
      f"{int(keep_mask_Q.sum())} px, {len(ref_circles)} circles @ r={KEEP_RADIUS:.0f}px "
      f"within [{MIN_BEAM_DISTANCE:.0f}, {MAX_BEAM_DISTANCE:.0f}]px of direct beam.")

In [ ]:
# ------------------------------------------------------------------------------
# 9. Virtual Imaging (BF & ADF Detector Placement)
# ------------------------------------------------------------------------------
expand_BF = 8.0  # Scaled up from 4.0
center = (Q_Nx // 2, Q_Ny // 2)      # (128, 128)
radius = bf_radius + expand_BF       # BF
radii = (bf_radius + expand_BF, 1e3) # ADF

# Position the detectors
selected_datacube.position_detector(mode='circle', geometry=(center, radius))
selected_datacube.position_detector(mode='annulus', geometry=(center, radii))

# Compute Virtual Images
im_bf = selected_datacube.get_virtual_image(mode='circle', geometry=(center, radius), name='bright_field')
im_adf = selected_datacube.get_virtual_image(mode='annulus', geometry=(center, radii), name='dark_field')

# Plot Virtual Images
fig_virt, ax_virt = plt.subplots(1, 2, figsize=(12, 6))
ax_virt[0].imshow(im_bf.data, cmap='gray')
ax_virt[0].set_title("Virtual Bright Field")
ax_virt[0].axis('off')

ax_virt[1].imshow(im_adf.data, cmap='gray')
ax_virt[1].set_title("Virtual Dark Field")
ax_virt[1].axis('off')

plt.tight_layout()

# Save and force display inline
save_path_virt = f"{save_prefix}virtual_images.png"
fig_virt.savefig(save_path_virt, bbox_inches='tight', dpi=150)
plt.close(fig_virt)

display(Image(filename=save_path_virt))


In [ ]:
# ------------------------------------------------------------------------------
# 10. Choose Scan Positions for Tuning Disk Detection Parameters
# ------------------------------------------------------------------------------
# Find scan positions that are inside our ROI mask
active_y, active_x = np.where(combined_mask)

if len(active_y) >= 6:
    # Select 6 points evenly distributed across the active region
    indices = np.linspace(0, len(active_y) - 1, 6, dtype=int)
    rxs = list(active_x[indices])
    rys = list(active_y[indices])
else:
    rxs = [W_scan // 2] * 6
    rys = [H_scan // 2] * 6

colors = ['r', 'limegreen', 'c', 'g', 'orange', 'violet']
print(f"Selected tuning scan positions (x, y): {list(zip(rxs, rys))}")

# Plot the chosen points on the Virtual ADF image
fig_pts, ax = plt.subplots(figsize=(6, 6))
ax.imshow(im_adf.data, cmap='gray')
ax.scatter(rxs, rys, c=colors, s=100, edgecolor='white', linewidth=1.5)
ax.set_title("Selected Tuning Positions on Virtual ADF")
ax.axis('on')

# Save and force display inline
save_path_pts = f"{save_prefix}tuning_positions.png"
fig_pts.savefig(save_path_pts, bbox_inches='tight', dpi=150)
plt.close(fig_pts)

display(Image(filename=save_path_pts))


In [ ]:
# ------------------------------------------------------------------------------
# 11. Test Disk Detection Parameters (show_image_grid)  [reference-filtered]
# ------------------------------------------------------------------------------
# Adjust parameters below in real-time to optimize disk detection
# Note: minPeakSpacing and edgeBoundary have been scaled for 256x256 resolution
detect_params = {
    'corrPower': 1.0,
    'sigma': 0,
    'edgeBoundary': 32,            # Scaled up from 16
    'minRelativeIntensity': 0.0001,
    'minAbsoluteIntensity': 0.0001,
    'minPeakSpacing': 35,           # Scaled up from 15
    'subpixel': 'poly',
    'upsample_factor': 8,
    'maxNumPeaks': 1000,
}

# Detect Bragg disks only on the selected tuning positions
disks_selected_raw = selected_datacube.find_Bragg_disks(
    data=(rys, rxs),
    template=probe_kernel,
    **detect_params,
)

# --- Filter with the reference keep-mask: remove peaks outside the reference circles (= Cell 9) ---
disks_selected = filter_disks_by_reference(disks_selected_raw, keep_mask_Q)

# Print counts before/after filtering
n_raw = sum(len(pl.data) for pl in disks_selected_raw)
n_flt = sum(len(pl.data) for pl in disks_selected)
print(f"Tuning detection: {n_raw} peaks -> {n_flt} peaks after reference filtering "
      f"(removed {n_raw - n_flt} outside reference circles).")

# Visualize the FILTERED detected disks on the diffraction patterns.
# Also draw the reference circles (keep-circles) to visually confirm the pass region.
def _get_annotate(i):
    # show_image_grid only supports get_x/get_y, so the circle overlay is drawn separately below.
    return None

grid = py4DSTEM.visualize.show_image_grid(
    get_ar=lambda i: selected_datacube.data[rys[i], rxs[i], :, :],
    H=2,
    W=3,
    axsize=(5, 5),
    intensity_range='minmax',
    vmin=0,
    vmax=0.05,
    scaling='power',
    power=0.2,
    get_bordercolor=lambda i: colors[i],
    get_x=lambda i: disks_selected[i].data['qx'],
    get_y=lambda i: disks_selected[i].data['qy'],
    get_pointcolors=lambda i: colors[i],
    open_circles=True,
    scale=500,
    returnfig=True,
)

# Faintly overlay the reference keep-circles on each subplot (to visually confirm the pass region)
try:
    fig_grid, axs_grid = grid
    axs_flat = np.array(axs_grid).ravel()
    for ax in axs_flat:
        for (rqx, rqy, r) in ref_circles:
            # matplotlib: (horizontal=qy, vertical=qx)
            ax.add_patch(plt.Circle((rqy, rqx), r, color='white',
                                    fill=False, linestyle=':', linewidth=0.8, alpha=0.5))
except Exception as _e:
    print(f"(overlay skipped: {_e})")

# Save and force display inline
save_path_grid = f"{save_prefix}detect_params_grid.png"
plt.savefig(save_path_grid, bbox_inches='tight', dpi=150)
plt.close()

display(Image(filename=save_path_grid))


In [ ]:
# Analyze the diffraction-pattern intensity of tuning point #0
sample_dp = selected_datacube.data[rys[0], rxs[0], :, :]
print("Min intensity (Min):", sample_dp.min())
print("Max intensity (Max):", sample_dp.max())
print("Mean intensity (Mean):", sample_dp.mean())

In [ ]:
# ------------------------------------------------------------------------------
# 12. Run Bragg Disk Detection on Entire ROI Dataset  [reference-filtered]
# ------------------------------------------------------------------------------
# After detection over the entire ROI, remove peaks outside the reference keep-mask.
# (Apply the same reference-circle criteria as tuning Cell 12 -> ensures consistency)
braggpeaks = selected_datacube.find_Bragg_disks(
    template=probe_kernel,
    **detect_params,
)
print("Bragg disk detection completed on the masked dataset.")

# --- Filter with the reference keep-mask ---
# BraggVectors.mask_in_Q(mask) "removes" peaks at positions where mask==True,
# so pass the inverse of the keep region (~keep_mask_Q) to remove peaks outside the reference circles.
def _count_bv_peaks(bv):
    v = bv._v_uncal
    return sum(len(v[rx, ry].data) for rx in range(v.shape[0]) for ry in range(v.shape[1]))

n_before = _count_bv_peaks(braggpeaks)
braggpeaks.mask_in_Q(~keep_mask_Q, update_inplace=True, returncalc=False)
n_after = _count_bv_peaks(braggpeaks)

print(f"Reference keep-mask filtering: {n_before} -> {n_after} peaks "
      f"({100*n_after/max(n_before,1):.1f}% retained, removed {n_before - n_after}).")


# ------------------------------------------------------------------------------
# 12b. Per-Reference-Circle Peak Thresholding  ->  weak/absent peak = NULL
# ------------------------------------------------------------------------------
# At each real-space point, clean up the peaks inside the reference circles:
#   - intensity < MIN_PEAK_INTENSITY -> treat as noise and remove (that circle is null at that point)
#   - multiple peaks in one circle -> keep only the single strongest one
# At the same time, build a per-(point x circle) "valid peak present" table (circle_peak_present)
# used in the strain step below to entirely exclude points where the g1/g2 circles are null.
#
# 'intensity' is the cross-correlation peak strength from find_Bragg_disks (small scale). Look at the distribution printed below and
# adjust MIN_PEAK_INTENSITY. (0 means no intensity cut, only duplicate removal)

MIN_PEAK_INTENSITY = 0.0    # <-- tuning point: peaks below this value are discarded as null

_ref_centers = np.array([[c[0], c[1]] for c in ref_circles], dtype=np.float64)  # (n,2)=(qx,qy)
_ref_radii   = np.array([c[2] for c in ref_circles], dtype=np.float64)
n_circles    = len(ref_circles)

_v = braggpeaks._v_uncal
R_Nx, R_Ny = _v.shape[0], _v.shape[1]

circle_peak_present   = np.zeros((R_Nx, R_Ny, n_circles), dtype=bool)
circle_peak_intensity = np.full((R_Nx, R_Ny, n_circles), np.nan, dtype=np.float64)

# Diagnostic: overall peak-intensity distribution
_all_int = []
for rx in range(R_Nx):
    for ry in range(R_Ny):
        d = _v[rx, ry].data
        if len(d):
            _all_int.append(np.asarray(d['intensity'], dtype=np.float64))
_all_int = np.concatenate(_all_int) if _all_int else np.array([0.0])
print("Per-peak intensity distribution (reference-filtered):")
for p in (1, 5, 25, 50, 75, 95, 99):
    print(f"   {p:2d}th pct = {np.percentile(_all_int, p):.6g}")
print(f"   min={_all_int.min():.6g}  max={_all_int.max():.6g}")
print(f">> MIN_PEAK_INTENSITY = {MIN_PEAK_INTENSITY:.6g} "
      f"({'no intensity cut (duplicates only)' if MIN_PEAK_INTENSITY <= 0 else 'peaks below this value are null'})\n")

def _apply_keep(pl, keep_flags):
    remove_mask = ~keep_flags
    if remove_mask.any():
        try:
            pl.remove(remove_mask)
        except Exception:
            pl.data = pl.data[keep_flags].copy()

n_kept_total = 0
n_removed_total = 0
for rx in range(R_Nx):
    for ry in range(R_Ny):
        d = _v[rx, ry].data
        if len(d) == 0:
            continue
        qx = np.asarray(d['qx'], dtype=np.float64)
        qy = np.asarray(d['qy'], dtype=np.float64)
        it = np.asarray(d['intensity'], dtype=np.float64)

        keep_flags = np.zeros(len(d), dtype=bool)
        for j in range(n_circles):
            dist = np.sqrt((qx - _ref_centers[j, 0])**2 + (qy - _ref_centers[j, 1])**2)
            in_circ = (dist <= _ref_radii[j]) & (it >= MIN_PEAK_INTENSITY)
            if not np.any(in_circ):
                continue                     # this circle is null at this point
            cand = np.where(in_circ)[0]
            best = cand[int(np.argmax(it[cand]))]
            keep_flags[best] = True
            circle_peak_present[rx, ry, j]   = True
            circle_peak_intensity[rx, ry, j] = it[best]

        n_kept_total    += int(keep_flags.sum())
        n_removed_total += int((~keep_flags).sum())
        _apply_keep(_v[rx, ry], keep_flags)

_slots = R_Nx * R_Ny * n_circles
print(f"Per-circle thresholding done: kept {n_kept_total} peaks, "
      f"removed {n_removed_total} (weak/duplicate).")
print(f"Filled circle-slots: {int(circle_peak_present.sum())}/{_slots} "
      f"({100.0*circle_peak_present.sum()/_slots:.1f}%). "
      f"Mean valid circles/point = {circle_peak_present.sum(axis=2).mean():.2f}/{n_circles}.")


In [ ]:
# ------------------------------------------------------------------------------
# 12.5 Verify Reference-Filtered Bragg Peaks (BVM overlay)
# ------------------------------------------------------------------------------
# Filtering is already done in Cell 14. Here we only check that the filter result
# matches the reference circles well by overlaying the reference keep-circles on the BVM.
assert 'keep_mask_Q' in globals(), "Run Cell 9 first to create keep_mask_Q."

bvm_after = braggpeaks.histogram(mode='raw')

# Adjust contrast so that even weak Bragg peaks are visible:
#  - stronger power scaling (0.3) to boost weak signals
#  - set vmax from the percentile of non-zero values so the central beam does not dominate vmax
img = bvm_after.data.astype(np.float32) ** 0.3
nz = img[img > 0]
vmax = np.percentile(nz, 99.5) if nz.size > 0 else img.max()

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img, cmap='inferno', vmin=0, vmax=vmax)
for (rqx, rqy, r) in ref_circles:
    # matplotlib: (horizontal=qy, vertical=qx)
    ax.add_patch(plt.Circle((rqy, rqx), r, color='cyan', fill=False,
                            linestyle='--', linewidth=1.5))
ax.set_title("Filtered BVM (raw) with Reference Keep-Circles")
ax.axis('on')
save_path_filt = f"{save_prefix}bvm_reference_filtered.png"
plt.savefig(save_path_filt, bbox_inches='tight', dpi=150)
plt.close()
display(Image(filename=save_path_filt))

In [ ]:
# --- Diagnostic ---
import numpy as np
bvm_raw = braggpeaks.histogram(mode='raw')
d = bvm_raw.data
print("dtype:", d.dtype, "shape:", d.shape)
print("min/max:", d.min(), d.max())
print("nonzero count:", (d>0).sum())
nz = d[d>0]
if nz.size:
    print("nonzero min/median/99.5pct/max:",
          nz.min(), np.median(nz), np.percentile(nz,99.5), nz.max())
_bvm_vmax = np.percentile(nz, 99.5) if nz.size else d.max()
print("computed _bvm_vmax:", _bvm_vmax)

In [ ]:
# ------------------------------------------------------------------------------
# 13. Centering and Origin Correction
# ------------------------------------------------------------------------------
def _show_bvm(bvm, title, save_name, point=None):
    """Draw directly with matplotlib instead of py4DSTEM.show to reliably secure contrast."""
    d = bvm.data.astype(np.float32)
    nz = d[d > 0]
    vmax = np.percentile(nz, 99.5) if nz.size > 0 else (d.max() if d.max() > 0 else 1.0)
    img = np.power(np.clip(d, 0, None), 0.25)
    vmax_p = vmax ** 0.25 if vmax > 0 else 1.0

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img, cmap='inferno', vmin=0, vmax=vmax_p)
    if point is not None:
        ax.plot(point[0], point[1], 'c+', markersize=15, markeredgewidth=2)
    ax.set_title(title)
    ax.axis('on')
    plt.savefig(save_name, bbox_inches='tight', dpi=150)
    plt.close()
    display(Image(filename=save_name))

# Plot raw Bragg Vector Map (BVM)
bvm_raw = braggpeaks.histogram(mode='raw')
_show_bvm(bvm_raw, "Raw BVM", f"{save_prefix}bvm_raw.png")

# Define center guess of diffraction space
center_guess = (Q_Nx // 2, Q_Ny // 2)
print(f"Initial center guess: {center_guess}")
_show_bvm(bvm_raw, "Raw BVM + center guess",
          f"{save_prefix}bvm_center_guess.png", point=center_guess)

# Measure origin and fit origin plane to correct for scan drift
origin_meas = braggpeaks.measure_origin(center_guess=center_guess)
# plot=False turns off fit_origin's automatic 6-panel diagnostic plot (so it does not overwrite the BVM figure)
qx0_fit, qy0_fit, qx0_residuals, qy0_residuals = braggpeaks.fit_origin(plot=False)

# Centered Bragg Vector Map (BVM)
bvm_centered = braggpeaks.histogram()
_show_bvm(bvm_centered, "Centered BVM", f"{save_prefix}bvm_centered.png")

In [ ]:
bvm_vis_params = {}

In [ ]:
# ------------------------------------------------------------------------------
# 13b. Helpers: exclude real-space points where the g1/g2 circles are null from strain
# ------------------------------------------------------------------------------
# Using circle_peak_present[y, x, j] built in Cell 12b (per-point valid-peak presence for each reference circle),
# a point is considered valid only if both circles corresponding to the chosen g1/g2 basis vectors are present.

def compute_basis_valid_mask(strainmap, g1_circle=None, g2_circle=None, verbose=True):
    fallback_shape = tuple(selected_datacube.data.shape[:2])
    if 'circle_peak_present' not in globals():
        if verbose:
            print("  [warn] circle_peak_present missing (re-run Cell 12b) -> mask not applied")
        return np.ones(fallback_shape, dtype=bool), None, None
    present = circle_peak_present

    def _get_vec(names):
        for n in names:
            v = getattr(strainmap, n, None)
            if v is not None:
                arr = np.asarray(v, dtype=np.float64).ravel()
                if arr.size >= 2:
                    return arr[:2]
        return None

    def _match(gvec):
        if gvec is None:
            return None
        pos = np.array([beam_qx, beam_qy], dtype=np.float64) + gvec   # absolute (qx,qy)
        dd = np.sqrt(((_ref_centers - pos) ** 2).sum(axis=1))
        return int(np.argmin(dd))

    cg1 = g1_circle if g1_circle is not None else _match(_get_vec(['g1']))
    cg2 = g2_circle if g2_circle is not None else _match(_get_vec(['g2']))

    if (cg1 is None or cg2 is None or cg1 >= present.shape[2] or cg2 >= present.shape[2]):
        if verbose:
            print("  [warn] g1/g2 -> reference circle matching failed -> mask not applied (all True). "
                  "If needed, specify manually via G1_CIRCLE_IDX/G2_CIRCLE_IDX.")
        return np.ones(present.shape[:2], dtype=bool), cg1, cg2

    vmask = present[:, :, cg1] & present[:, :, cg2]
    if verbose:
        print(f"  basis validity: g1->circle#{cg1}, g2->circle#{cg2}; "
              f"valid points = {int(vmask.sum())}/{vmask.size} "
              f"({100.0*vmask.mean():.1f}%)")
    return vmask, cg1, cg2


def apply_basis_null(strainmap, vmask):
    """Set the strain values of invalid (g1/g2-missing) points to NaN and the mask channel to 0."""
    invalid = ~np.asarray(vmask, dtype=bool)
    data = strainmap.data
    nch = data.shape[0]
    for k in range(min(4, nch)):        # 0=exx 1=eyy 2=exy 3=theta
        data[k][invalid] = np.nan
    if nch > 4:                          # 4=mask
        data[4][invalid] = 0.0
    return int(invalid.sum())

print("Basis-null helpers ready: compute_basis_valid_mask(), apply_basis_null().")


In [ ]:
# ------------------------------------------------------------------------------
# 14. Strain Mapping: Loop over All Basis Vector Combinations (g1 < g2, 1 to 5)
# ------------------------------------------------------------------------------
import itertools

strainmap = py4DSTEM.StrainMap(braggvectors=braggpeaks)

g_candidates = [1, 2, 3, 4, 5]
basis_combinations = list(itertools.combinations(g_candidates, 2))
print(f"Testing {len(basis_combinations)} combinations: {basis_combinations}")

for g1, g2 in basis_combinations:
    print("\n" + "="*80)
    print(f"Processing Basis Vector Pair: g1={g1}, g2={g2}")
    print("="*80)

    try:
        # 1) Select basis vectors
        strainmap.choose_basis_vectors(
            index_g1=g1,
            index_g2=g2,
            minSpacing=10,
            minAbsoluteIntensity=1000,
            maxNumPeaks=100,
            vis_params=bvm_vis_params,
        )

        # 2) Fit basis vectors
        strainmap.fit_basis_vectors(
            max_peak_spacing=4.0,
            vis_params=bvm_vis_params,
            returncalc=True
        )

        # 2b) Exclude points where the g1/g2 circles are null from strain (null instead of noise)
        vmask, _cg1, _cg2 = compute_basis_valid_mask(strainmap, verbose=True)

        # 3) Compute and visualize strain maps
        try:
            import cmasher as cmr
            colormap = cmr.holly
        except ImportError:
            colormap = 'RdBu_r'

        strainmap.get_strain(
            mask=vmask,
            vrange_exx=[-5.0, 5.0],
            vrange_exy=[-5.0, 5.0],
            vrange_eyy=[-5.0, 5.0],
            vrange_theta=[-5.0, 5.0],
            cmap=colormap,
            cmap_theta=colormap
        )
        apply_basis_null(strainmap, vmask)   # also set the excluded points' values to NaN

        # Save and force display inline
        save_path_strain_comb = f"{save_prefix}strain_maps_g{g1}_g{g2}.png"
        plt.savefig(save_path_strain_comb, bbox_inches='tight', dpi=150)
        plt.close()
        print(f"[Result] Strain map saved to: {save_path_strain_comb}")
        display(Image(filename=save_path_strain_comb))

    except Exception as e:
        print(f"Error fitting combination g1={g1}, g2={g2}: {e}")
        plt.close()


In [ ]:
# ------------------------------------------------------------------------------
# 15. Final Strain Map with Chosen Basis Vectors (g1=4, g2=5)
# ------------------------------------------------------------------------------
# Recompute and plot only the combination adopted from the results checked in the loop above.
g1, g2 = 1,4

# If the g1/g2 auto-matching is wrong, specify the reference circle indices directly below (integer). None means auto.
G1_CIRCLE_IDX = None
G2_CIRCLE_IDX = None

# Create a new StrainMap with the adopted combination (start fresh since the loop may leave the last combination's state)
strainmap = py4DSTEM.StrainMap(braggvectors=braggpeaks)

# Prepare colormap
try:
    import cmasher as cmr
    colormap = cmr.holly
except ImportError:
    colormap = 'RdBu_r'

print(f"Computing final strain map for chosen basis vectors: g1={g1}, g2={g2}")

# 1) Select basis vectors
strainmap.choose_basis_vectors(
    index_g1=g1,
    index_g2=g2,
    minSpacing=10,
    minAbsoluteIntensity=1000,
    maxNumPeaks=100,
    vis_params=bvm_vis_params,
)

# 2) Fit basis vectors
strainmap.fit_basis_vectors(
    max_peak_spacing=4.0,
    vis_params=bvm_vis_params,
    returncalc=True,
)

# 2b) Exclude points where the g1/g2 circles are null from strain (null instead of noise)
vmask, cg1, cg2 = compute_basis_valid_mask(strainmap, G1_CIRCLE_IDX, G2_CIRCLE_IDX)

# 3) Compute and visualize strain maps
strainmap.get_strain(
    mask=vmask,
    vrange_exx=[-5.0, 5.0],
    vrange_exy=[-5.0, 5.0],
    vrange_eyy=[-5.0, 5.0],
    vrange_theta=[-5.0, 5.0],
    cmap=colormap,
    cmap_theta=colormap,
)
apply_basis_null(strainmap, vmask)

# Save and force display inline
save_path_strain_final = f"{save_prefix}strain_maps_final_g{g1}_g{g2}.png"
plt.savefig(save_path_strain_final, bbox_inches='tight', dpi=150)
plt.close()
print(f"[Result] Final strain map saved to: {save_path_strain_final}")
print(f"[Info] Nulled points (g1/g2 circle missing): {int((~vmask).sum())} / {vmask.size}")
display(Image(filename=save_path_strain_final))


In [ ]:
# ------------------------------------------------------------------------------
# 16. exx Region-Split Average DPs & Color Overlay (High-exx: red / Low-exx: green)
# ------------------------------------------------------------------------------
# Compute the mean DP for the exx >= +0.5% (tension) and exx <= -0.5% (compression) regions respectively,
# and overlay in red(+)/green(-). Darken the background and show only the spots brightly.
import numpy as np

# --- Threshold (strain is stored as a fraction: 0.5% = 0.005) ---
EXX_THRESH = 0.005    # ±0.5%

# --- Contrast parameters (tuning points) ---
BG_PCT  = 40.0    # values at or below this percentile are treated as background and zeroed (higher = darker)
TOP_PCT = 99.5    # upper brightness clipping threshold
GAMMA   = 0.6     # gamma (lower boosts weak spots; 0.5~0.8 recommended)

# --- Unpack strainmap.data:  [0]=exx [1]=eyy [2]=exy [3]=theta [4]=mask ---
exx_map     = strainmap.data[0]
strain_mask = strainmap.data[4] > 0.5    # valid (fit-successful) pixels only

region_pos = strain_mask & (exx_map >=  EXX_THRESH)   # tension (+) region
region_neg = strain_mask & (exx_map <= -EXX_THRESH)   # compression (-) region

n_pos = int(region_pos.sum())
n_neg = int(region_neg.sum())
print(f"exx >= +{EXX_THRESH*100:.1f}% : {n_pos} pixels")
print(f"exx <= -{EXX_THRESH*100:.1f}% : {n_neg} pixels")

assert exx_map.shape == selected_datacube.data.shape[:2], \
    f"shape mismatch: strain {exx_map.shape} vs cube {selected_datacube.data.shape[:2]}"

# --- Mean DP of each region ---
def _average_dp(region):
    ys, xs = np.where(region)
    if len(ys) == 0:
        return None
    acc = np.zeros(selected_datacube.data.shape[2:], dtype=np.float64)
    for y, x in zip(ys, xs):
        acc += selected_datacube.data[y, x, :, :]
    return acc / len(ys)

dp_pos = _average_dp(region_pos)
dp_neg = _average_dp(region_neg)

# --- Suppress background and normalize only the spots ---
def _norm_spots(d, bg_pct=BG_PCT, top_pct=TOP_PCT, gamma=GAMMA):
    if d is None:
        return None
    d = np.clip(d.astype(np.float32), 0, None)
    floor = np.percentile(d, bg_pct)
    v = np.clip(d - floor, 0, None)
    vmax = np.percentile(v[v > 0], top_pct) if np.any(v > 0) else 1.0
    v = np.clip(v / vmax, 0, 1) if vmax > 0 else v
    return v ** gamma

r = _norm_spots(dp_pos)   # red = tension (+)
g = _norm_spots(dp_neg)   # green = compression (-)

# RGB composite (overlapping spots become red+green = yellow)
shape2d = (r if r is not None else g).shape
rgb = np.zeros((*shape2d, 3), dtype=np.float32)
if r is not None: rgb[..., 0] = r   # R channel
if g is not None: rgb[..., 1] = g   # G channel

red_only = np.zeros_like(rgb)
if r is not None: red_only[..., 0] = r
green_only = np.zeros_like(rgb)
if g is not None: green_only[..., 1] = g

# --- For real-space region locations (only selected pixels colored, rest black) ---
dark_bg = np.zeros((*strain_mask.shape, 3), dtype=np.float32)

pos_img = dark_bg.copy()
pos_img[region_pos] = [1.0, 0.0, 0.0]          # (+) = red

neg_img = dark_bg.copy()
neg_img[region_neg] = [0.0, 1.0, 0.0]          # (-) = green

both_img = dark_bg.copy()
both_img[region_pos] = [1.0, 0.0, 0.0]
both_img[region_neg] = [0.0, 1.0, 0.0]

# --- Visualization (2 rows x 3 cols, compact) ---
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))

# [Top row] only selected real-space pixels in color, rest darkened
axes[0, 0].imshow(pos_img)
axes[0, 0].set_title(f"(+) region  (n={n_pos})", fontsize=10)
axes[0, 0].axis('off')

axes[0, 1].imshow(both_img)
axes[0, 1].set_title("(+) red / (-) green", fontsize=10)
axes[0, 1].axis('off')

axes[0, 2].imshow(neg_img)
axes[0, 2].set_title(f"(-) region  (n={n_neg})", fontsize=10)
axes[0, 2].axis('off')

# [Bottom row] red-only / overlay / green-only DP
axes[1, 0].imshow(red_only)
axes[1, 0].set_title(f"exx >= +{EXX_THRESH*100:.1f}%  (red)", fontsize=10)
axes[1, 0].axis('off')

axes[1, 1].imshow(rgb)
axes[1, 1].set_title("Overlay:  (+) red / (-) green / shared yellow", fontsize=10)
axes[1, 1].axis('off')

axes[1, 2].imshow(green_only)
axes[1, 2].set_title(f"exx <= -{EXX_THRESH*100:.1f}%  (green)", fontsize=10)
axes[1, 2].axis('off')

plt.tight_layout()
save_path_exx_dp = f"{save_prefix}exx_region_DP_overlay.png"
plt.savefig(save_path_exx_dp, bbox_inches='tight', dpi=150)
plt.close()
display(Image(filename=save_path_exx_dp))

In [ ]:
# ------------------------------------------------------------------------------
# 15b. Histograms of Final Strain Components (exx, eyy, exy, theta)
# ------------------------------------------------------------------------------
# Plot the distribution of the strain components of the final strainmap (g1=1, g2=5) obtained in Cell 15 (= Cell 21).
# Exclude NaN (= basis-null / fit-failed) points.
import numpy as np
import matplotlib.pyplot as plt

# Valid-pixel mask: mask channel (>0.5) & each component non-NaN
valid = strainmap.data[4] > 0.5

XLIM = (-5, 5)   # common range for the four components

# (name, array, unit, bar color)  -- exy is shear strain (not an angle); only theta is an angle (deg)
comps = [
    ("exx",   strainmap.data[0], "%",   "#4C72B0"),  # muted blue
    ("eyy",   strainmap.data[1], "%",   "#55A868"),  # muted green
    ("exy",   strainmap.data[2], "%",   "#C44E52"),  # muted red
    ("theta", strainmap.data[3], "deg", "#8172B3"),  # muted purple
]

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "axes.edgecolor":  "#444444",
    "axes.linewidth":  0.9,
    "axes.titlesize":  12,
    "axes.titleweight":"bold",
    "axes.labelsize":  11,
})

fig, axes = plt.subplots(2, 2, figsize=(11, 8), facecolor="white")
axes = axes.ravel()

for ax, (name, arr, unit, color) in zip(axes, comps):
    vals = arr[valid & np.isfinite(arr)].ravel()
    if name == "theta":
        vals = np.degrees(vals)          # rad -> deg
    else:
        vals = vals * 100.0              # fraction -> %

    if vals.size == 0:
        ax.set_title(f"{name}: no valid data")
        ax.axis("off")
        continue

    ax.hist(vals, bins=60, range=XLIM,
            color=color, edgecolor="white", linewidth=0.4, alpha=0.9)
    ax.set_xlim(XLIM)
    ax.set_xticks(np.arange(XLIM[0], XLIM[1] + 1, 1))

    mean, std, med = np.mean(vals), np.std(vals), np.median(vals)
    ax.axvline(mean, color="#2b2b2b", ls="--", lw=1.3, label=f"mean = {mean:.3f}")
    ax.axvline(med,  color="#2b2b2b", ls=":",  lw=1.3, label=f"median = {med:.3f}")

    ax.set_title(f"{name}   (n = {vals.size:,},  σ = {std:.3f} {unit})")
    ax.set_xlabel(f"{name} [{unit}]")
    ax.set_ylabel("Count")

    # Clean style: remove top/right spines, faint horizontal grid
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", ls="-", lw=0.6, color="#dddddd", alpha=0.8)
    ax.set_axisbelow(True)                 # put grid behind bars
    ax.tick_params(length=3, colors="#444444")
    ax.legend(fontsize=8, frameon=False)

fig.suptitle(f"Strain Component Distributions  (final basis  g1={g1}, g2={g2})",
             fontsize=14, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])

save_path_hist = f"{save_prefix}strain_histograms_g{g1}_g{g2}.png"
plt.savefig(save_path_hist, bbox_inches="tight", dpi=200, facecolor="white")
plt.show()
print(f"[Result] Strain histograms saved to: {save_path_hist}")